In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path = '/content/drive/MyDrive/ПИ, ПИС (испр).xlsx'

drive_path = '/content/drive/MyDrive/'
files = [f for f in Path(drive_path).iterdir() if f.suffix == '.xlsx']

In [4]:
def parse_disciplines(file_path):
    df_raw = pd.read_excel(file_path, sheet_name='Дисциплины', header=None, dtype=str)

    start_row = None
    for idx, row in df_raw.iterrows():
        first_cell = str(row[0]) if pd.notna(row[0]) else ""
        if 'Наименование блоков' in first_cell:
            start_row = idx + 1
            break

    if start_row is None:
        for idx, row in df_raw.iterrows():
            first_cell = str(row[0]) if pd.notna(row[0]) else ""
            if 'Б.1' in first_cell or 'Блок 1' in first_cell:
                start_row = idx
                break

    if start_row is None:
        raise ValueError("Не удалось найти начало данных о дисциплинах")

    disciplines = []
    skip_keywords = ['Блок', 'Цикл', 'ИТОГО', 'Всего на', 'Обязательная часть', 'Часть, формируемая']

    for idx in range(start_row, len(df_raw)):
        row = df_raw.iloc[idx]
        disc_name = str(row[0]).strip() if pd.notna(row[0]) else ""

        if not disc_name or any(kw in disc_name for kw in skip_keywords):
            continue

        if disc_name and disc_name[0].isdigit() and '.' in disc_name[:5]:
            continue

        disc_info = {
            'дисциплина': disc_name,
            'кафедра': row[2] if len(row) > 2 and pd.notna(row[2]) else "",
            'компетенции': row[3] if len(row) > 3 and pd.notna(row[3]) else "",
            'экзамен': row[4] if len(row) > 4 and pd.notna(row[4]) else "",
            'зачет': row[5] if len(row) > 5 and pd.notna(row[5]) else "",
            'зачет_с_оценкой': row[6] if len(row) > 6 and pd.notna(row[6]) else "",
            'курсовая_работа': row[7] if len(row) > 7 and pd.notna(row[7]) else "",
            'курсовой_проект': row[8] if len(row) > 8 and pd.notna(row[8]) else "",
            'зачетные_единицы': row[13] if len(row) > 13 and pd.notna(row[13]) else 0,
            'часов_всего': row[14] if len(row) > 14 and pd.notna(row[14]) else 0,
            'аудиторных': row[15] if len(row) > 15 and pd.notna(row[15]) else 0,
            'лекции_всего': row[16] if len(row) > 16 and pd.notna(row[16]) else 0,
            'семинары_всего': row[17] if len(row) > 17 and pd.notna(row[17]) else 0,
            'самостоятельная_всего': row[18] if len(row) > 18 and pd.notna(row[18]) else 0,
        }

        for sem in range(1, 9):
            base_col = 20 + (sem - 1) * 4
            for i, suffix in enumerate(['з.е.', 'лекции', 'семинары', 'самостоятельная']):
                col_idx = base_col + i
                disc_info[f'{sem}_семестр_{suffix}'] = row[col_idx] if col_idx < len(row) and pd.notna(row[col_idx]) else 0

        disciplines.append(disc_info)
        if len(disciplines) > 200:
            break

    df = pd.DataFrame(disciplines)

    numeric_cols = ['зачетные_единицы', 'часов_всего', 'аудиторных', 'лекции_всего', 'семинары_всего', 'самостоятельная_всего']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    for sem in range(1, 9):
        for suffix in ['з.е.', 'лекции', 'семинары', 'самостоятельная']:
            col = f'{sem}_семестр_{suffix}'
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    return df

In [5]:
def parse_schedule(file_path):
    """
    Парсит лист "График прохождения" и извлекает распределение недель.
    """
    df_raw = pd.read_excel(file_path, sheet_name='График прохождения', header=None, dtype=str)

    schedule_data = []

    # Поиск строк, где в первой колонке указан номер курса (1-4)
    for idx, row in df_raw.iterrows():
        first_cell = str(row[0]) if pd.notna(row[0]) else ""

        # Проверяем, является ли значение номером курса
        if first_cell.strip() in ['1', '2', '3', '4'] and len(first_cell.strip()) == 1:
            course_num = int(first_cell)
            weeks = {'курс': course_num}

            # Извлекаем данные по неделям (колонки с 1 по 52)
            for i in range(1, min(53, len(row))):
                if pd.notna(row[i]) and str(row[i]).strip():
                    weeks[f'неделя_{i}'] = str(row[i]).strip()
                else:
                    weeks[f'неделя_{i}'] = ''

            schedule_data.append(weeks)

    df_schedule = pd.DataFrame(schedule_data)
    print(f"Найдено данных для {len(df_schedule)} курсов")

    return df_schedule

In [6]:
def parse_competence_matrix(file_path):
    """
    Парсит лист "Матрица" и преобразует матрицу в длинный формат.
    """
    df_raw = pd.read_excel(file_path, sheet_name='Матрица', header=None, dtype=str)

    # Поиск строки с заголовками компетенций (например, ПКН-1)
    comp_row_idx = None
    for idx, row in df_raw.iterrows():
        row_str = ' '.join([str(x) for x in row.values if pd.notna(x)])
        if 'ПКН-1' in row_str:
            comp_row_idx = idx
            break

    if comp_row_idx is None:
        print("Матрица компетенций не найдена")
        return None, []

    # Извлечение списка компетенций из строки заголовка
    competencies = []
    for i in range(1, len(df_raw.columns)):
        val = df_raw.iloc[comp_row_idx, i]
        if pd.notna(val) and str(val).strip():
            competencies.append(str(val).strip())

    # Сбор данных матрицы (дисциплины -> компетенции)
    matrix_data = []
    for idx in range(comp_row_idx + 1, len(df_raw)):
        row = df_raw.iloc[idx]
        disc_name = str(row[0]) if pd.notna(row[0]) else ""
        disc_name = disc_name.strip()

        # Пропуск пустых строк
        if not disc_name or disc_name == 'nan':
            continue

        # Для каждой компетенции проверяем наличие связи с дисциплиной
        for comp_idx, comp in enumerate(competencies):
            if comp_idx + 1 < len(row) and pd.notna(row[comp_idx + 1]):
                semesters = str(row[comp_idx + 1]).strip()
                if semesters and semesters != 'nan':
                    matrix_data.append({
                        'дисциплина': disc_name,
                        'компетенция': comp,
                        'семестры': semesters
                    })

    df_matrix = pd.DataFrame(matrix_data)
    print(f"Найдено {len(matrix_data)} связей дисциплин с компетенциями")

    return df_matrix, competencies

In [7]:
def parse_full_curriculum(file_path):
    """
    Парсит все листы учебного плана и объединяет в единый датафрейм.
    """
    print("Парсинг дисциплин...")
    df_disciplines = parse_disciplines(file_path)

    print("Парсинг графика прохождения...")
    df_schedule = parse_schedule(file_path)

    print("Парсинг матрицы компетенций...")
    df_matrix, competencies = parse_competence_matrix(file_path)

    # Создание длинного формата для анализа по семестрам
    sem_data = []

    for _, disc in df_disciplines.iterrows():
        disc_name = disc['дисциплина']

        # Проход по всем семестрам (1-8)
        for sem in range(1, 9):
            ze_col = f'{sem}_семестр_з.е.'

            # Добавляем дисциплину только если в семестре есть зачетные единицы
            if ze_col in disc and disc[ze_col] > 0:
                sem_data.append({
                    'дисциплина': disc_name,
                    'семестр': sem,
                    'курс': (sem + 1) // 2,  # Преобразование семестра в курс
                    'зачетные_единицы': disc[ze_col],
                    'лекции_часов': disc.get(f'{sem}_семестр_лекции', 0),
                    'семинары_часов': disc.get(f'{sem}_семестр_семинары', 0),
                    'самостоятельная_часов': disc.get(f'{sem}_семестр_самостоятельная', 0)
                })

    df_by_semester = pd.DataFrame(sem_data)

    print("\nПарсинг успешно завершен!")

    return {
        'дисциплины': df_disciplines,
        'дисциплины_по_семестрам': df_by_semester,
        'график_прохождения': df_schedule,
        'матрица_компетенций': df_matrix
    }

In [8]:
# Запускаем парсинг
print("Начинаем парсинг учебного плана...")
print("=" * 60)

try:
    parsed_data = parse_full_curriculum(file_path)

    # Сохраняем результаты
    df_disciplines = parsed_data['дисциплины']
    df_by_semester = parsed_data['дисциплины_по_семестрам']
    df_schedule = parsed_data['график_прохождения']
    df_matrix = parsed_data['матрица_компетенций']

    print(f"\nИтоговые данные:")
    print(f"   - Дисциплин: {len(df_disciplines)}")
    print(f"   - Записей по семестрам: {len(df_by_semester)}")
    if df_matrix is not None:
        print(f"   - Связей с компетенциями: {len(df_matrix)}")

except Exception as e:
    print(f"\nОшибка при парсинге: {e}")
    import traceback
    traceback.print_exc()

Начинаем парсинг учебного плана...
Парсинг дисциплин...
Парсинг графика прохождения...
Найдено данных для 4 курсов
Парсинг матрицы компетенций...
Найдено 94 связей дисциплин с компетенциями

Парсинг успешно завершен!

Итоговые данные:
   - Дисциплин: 100
   - Записей по семестрам: 162
   - Связей с компетенциями: 94


In [9]:
# Просмотр результатов
print("=" * 80)
print("РЕЗУЛЬТАТЫ ПАРСИНГА")
print("=" * 80)

if len(df_disciplines) > 0:
    print(f"\nДисциплины (всего: {len(df_disciplines)})")
    print("-" * 40)
    display(df_disciplines[['дисциплина', 'кафедра', 'зачетные_единицы', 'компетенции']].head(20))

if len(df_by_semester) > 0:
    print(f"\nРаспределение по семестрам (всего записей: {len(df_by_semester)})")
    print("-" * 40)
    display(df_by_semester.head(20))

if len(df_schedule) > 0:
    print(f"\nГрафик прохождения")
    print("-" * 40)
    display(df_schedule)

if df_matrix is not None and len(df_matrix) > 0:
    print(f"\nМатрица компетенций (всего связей: {len(df_matrix)})")
    print("-" * 40)
    display(df_matrix.head(20))

РЕЗУЛЬТАТЫ ПАРСИНГА

Дисциплины (всего: 100)
----------------------------------------


,дисциплина,кафедра,зачетные_единицы,компетенции
0,Б.1,,207,
1,Б.1.1,,102,
2,Б.1.1.1,,38,
3,Б.1.1.1.1,КАДиМО,1,"УК-12,УК-1"
4,Б.1.1.1.2,КГН,4,"УК-1,УК-15"
5,Б.1.1.1.3,БЖД,2,УК-7
6,Б.1.1.1.4,ФВ,2,УК-6
7,Б.1.1.1.5,КАЯиПК,8,УК-3
8,Б.1.1.1.6,КАЯиПК,4,"УК-3,УК-2"
9,Б.1.1.1.7,КГН,4,"УК-1,УК-9"



Распределение по семестрам (всего записей: 162)
----------------------------------------


,дисциплина,семестр,курс,зачетные_единицы,лекции_часов,семинары_часов,самостоятельная_часов
0,Б.1,1,1,27,118,338,552
1,Б.1,2,1,32,146,392,686
2,Б.1,3,2,58,23,96,290
3,Б.1,4,2,658,25,112,306
4,Б.1,5,3,626,64,31,180
5,Б.1,6,3,340,722,24,128
6,Б.1,7,4,274,660,23,19
7,Б.1,8,4,112,206,510,0
8,Б.1.1,1,1,21,86,270,436
9,Б.1.1,2,1,26,114,324,498



График прохождения
----------------------------------------


,курс,неделя_1,неделя_2,неделя_3,неделя_4,неделя_5,неделя_6,неделя_7,неделя_8,неделя_9,...,неделя_43,неделя_44,неделя_45,неделя_46,неделя_47,неделя_48,неделя_49,неделя_50,неделя_51,неделя_52
0,1,,,,,,,,,,...,ПА,К,К,К,К,К,К,К,К,К
1,2,,,,,,,,,,...,ПА,К,К,К,К,К,К,К,К,К
2,3,,,,,,,,,,...,ПА,К,К,К,К,К,К,К,К,К
3,4,,,,,,,,,,...,ГИА,ГИА,К,К,К,К,К,К,К,К



Матрица компетенций (всего связей: 94)
----------------------------------------


,дисциплина,компетенция,семестры
0,Алгебра и анализ,ПКН-7,"1, 2"
1,Алгоритмы и структуры данных в языке Python,ПКН-2,"1, 2"
2,Алгоритмы и структуры данных в языке Python,ПКН-3,"1, 2"
3,Банковские информационные системы,ПКП-2,5
4,Безопасность жизнедеятельности,УК-7,1
5,Введение в специальность,ПКН-5,1
6,Введение в специальность,УК-8,1
7,Глубокое обучение в финансах,ПКН-4,7
8,Глубокое обучение в финансах,ПКП-6,7
9,Дискретная математика,ПКН-1,"1, 2"


In [10]:
print("=" * 80)
print("АНАЛИТИЧЕСКАЯ СВОДКА")
print("=" * 80)

# Проверка наличия данных о дисциплинах
if len(df_disciplines) == 0:
    print("Нет данных о дисциплинах для анализа")
else:
    print(f"Найдено {len(df_disciplines)} дисциплин")

# Нагрузка по семестрам
if len(df_by_semester) > 0:
    print("\nНагрузка по семестрам:")
    print("-" * 50)
    load_by_sem = df_by_semester.groupby('семестр').agg({
        'зачетные_единицы': 'sum',
        'лекции_часов': 'sum',
        'семинары_часов': 'sum',
        'самостоятельная_часов': 'sum'
    }).round(2)
    display(load_by_sem)

    # Нагрузка по курсам
    print("\nНагрузка по курсам:")
    print("-" * 50)
    load_by_course = df_by_semester.groupby('курс').agg({
        'зачетные_единицы': 'sum',
        'лекции_часов': 'sum',
        'семинары_часов': 'sum',
        'самостоятельная_часов': 'sum'
    }).round(2)
    display(load_by_course)
else:
    print("\nНет данных о распределении по семестрам")

# Распределение по кафедрам
if 'кафедра' in df_disciplines.columns and df_disciplines['кафедра'].notna().any():
    print("\nРаспределение дисциплин по кафедрам:")
    print("-" * 50)
    dept_stats = df_disciplines[df_disciplines['кафедра'] != ''].groupby('кафедра').size().sort_values(ascending=False)
    if len(dept_stats) > 0:
        display(dept_stats)
    else:
        print("Нет данных о кафедрах")
else:
    print("\nНет данных о кафедрах")

# Распределение по видам контроля
print("\nРаспределение по видам контроля:")
print("-" * 50)

control_list = []

if 'экзамен' in df_disciplines.columns:
    for _, row in df_disciplines.iterrows():
        if pd.notna(row['экзамен']) and row['экзамен'] != '' and row['экзамен'] != 'nan':
            control_list.append(f"экзамен:{row['экзамен']}")

if 'зачет' in df_disciplines.columns:
    for _, row in df_disciplines.iterrows():
        if pd.notna(row['зачет']) and row['зачет'] != '' and row['зачет'] != 'nan':
            control_list.append(f"зачет:{row['зачет']}")

if 'зачет_с_оценкой' in df_disciplines.columns:
    for _, row in df_disciplines.iterrows():
        if pd.notna(row['зачет_с_оценкой']) and row['зачет_с_оценкой'] != '' and row['зачет_с_оценкой'] != 'nan':
            control_list.append(f"зачет_с_оценкой:{row['зачет_с_оценкой']}")

if 'курсовая_работа' in df_disciplines.columns:
    for _, row in df_disciplines.iterrows():
        if pd.notna(row['курсовая_работа']) and row['курсовая_работа'] != '' and row['курсовая_работа'] != 'nan':
            control_list.append(f"курсовая:{row['курсовая_работа']}")

if 'курсовой_проект' in df_disciplines.columns:
    for _, row in df_disciplines.iterrows():
        if pd.notna(row['курсовой_проект']) and row['курсовой_проект'] != '' and row['курсовой_проект'] != 'nan':
            control_list.append(f"курсовой_проект:{row['курсовой_проект']}")

if control_list:
    control_series = pd.Series(control_list)
    control_stats = control_series.value_counts().head(20)
    display(control_stats)
else:
    print("Нет данных о видах контроля")

# Статистика по компетенциям
if df_matrix is not None and len(df_matrix) > 0:
    print("\nТоп-15 компетенций по частоте встречаемости:")
    print("-" * 50)
    comp_stats = df_matrix['компетенция'].value_counts()
    display(comp_stats.head(15))
else:
    print("\nНет данных о компетенциях")

# Общая статистика
print("\nОбщая статистика:")
print("-" * 50)
print(f"Всего дисциплин: {len(df_disciplines)}")

if df_matrix is not None and len(df_matrix) > 0:
    print(f"Всего компетенций: {len(df_matrix['компетенция'].unique())}")
    print(f"Среднее количество компетенций на дисциплину: {df_matrix.groupby('дисциплина').size().mean():.2f}")
else:
    print("Нет данных о компетенциях")

if 'кафедра' in df_disciplines.columns:
    dept_count = len(df_disciplines[df_disciplines['кафедра'] != '']['кафедра'].unique())
    print(f"Всего кафедр: {dept_count}")

if 'зачетные_единицы' in df_disciplines.columns:
    total_ects = pd.to_numeric(df_disciplines['зачетные_единицы'], errors='coerce').sum()
    print(f"Общее количество зачетных единиц: {total_ects:.0f}")
elif 'зачетные_единицы_всего' in df_disciplines.columns:
    total_ects = pd.to_numeric(df_disciplines['зачетные_единицы_всего'], errors='coerce').sum()
    print(f"Общее количество зачетных единиц: {total_ects:.0f}")

# Статистика по семестрам
if len(df_by_semester) > 0:
    print("\nСтатистика по семестрам:")
    print("-" * 50)
    for sem in range(1, 9):
        sem_data = df_by_semester[df_by_semester['семестр'] == sem]
        if len(sem_data) > 0:
            total_ects = sem_data['зачетные_единицы'].sum()
            total_aud = sem_data['лекции_часов'].sum() + sem_data['семинары_часов'].sum()
            print(f"Семестр {sem}: {len(sem_data)} дисциплин, {total_ects:.1f} з.е., {total_aud:.0f} аудиторных часов")
        else:
            print(f"Семестр {sem}: нет данных")

АНАЛИТИЧЕСКАЯ СВОДКА
Найдено 100 дисциплин

Нагрузка по семестрам:
--------------------------------------------------


,зачетные_единицы,лекции_часов,семинары_часов,самостоятельная_часов
семестр,,,,
1,108,472,1352,2208
2,136,584,1704,2744
3,235,92,390,1166
4,2632,88,384,1088
5,2504,149,81,476
6,1672,3304,99,502
7,1622,3744,117,79
8,688,1254,3062,0



Нагрузка по курсам:
--------------------------------------------------


,зачетные_единицы,лекции_часов,семинары_часов,самостоятельная_часов
курс,,,,
1,244,1056,3056,4952
2,2867,180,774,2254
3,4176,3453,180,978
4,2310,4998,3179,79



Распределение дисциплин по кафедрам:
--------------------------------------------------


...


Распределение по видам контроля:
--------------------------------------------------


,count
зачет:7,14
зачет:6,13
зачет:3,8
экзамен:6,7
зачет:1,7
зачет:5,6
зачет_с_оценкой:2,5
экзамен:4,5
экзамен:5,5
курсовой_проект:2,4



Топ-15 компетенций по частоте встречаемости:
--------------------------------------------------


,count
компетенция,
ПКП-6,10
ПКП-3,9
ПКП-2,7
ПКН-2,7
ПКП-4,6
ПКН-1,5
ПКН-7,5
ПКН-4,5
УК-1,4



Общая статистика:
--------------------------------------------------
Всего дисциплин: 100
Всего компетенций: 29
Среднее количество компетенций на дисциплину: 1.45
Всего кафедр: 12
Общее количество зачетных единиц: 1032

Статистика по семестрам:
--------------------------------------------------
Семестр 1: 18 дисциплин, 108.0 з.е., 1824 аудиторных часов
Семестр 2: 19 дисциплин, 136.0 з.е., 2288 аудиторных часов
Семестр 3: 22 дисциплин, 235.0 з.е., 482 аудиторных часов
Семестр 4: 15 дисциплин, 2632.0 з.е., 472 аудиторных часов
Семестр 5: 16 дисциплин, 2504.0 з.е., 230 аудиторных часов
Семестр 6: 22 дисциплин, 1672.0 з.е., 3403 аудиторных часов
Семестр 7: 29 дисциплин, 1622.0 з.е., 3861 аудиторных часов
Семестр 8: 21 дисциплин, 688.0 з.е., 4316 аудиторных часов


In [11]:
# Сохраняем результаты в CSV файлы на Google Drive
save_results = True  # False, если не сохранять

if save_results:
    output_dir = '/content/drive/MyDrive/Curriculum_Parser_Results/'

    # Создаем папку для результатов
    os.makedirs(output_dir, exist_ok=True)

    print("Сохранение результатов...")

    # Сохраняем каждый датафрейм
    df_disciplines.to_csv(f'{output_dir}disciplines.csv', index=False, encoding='utf-8-sig')
    df_by_semester.to_csv(f'{output_dir}disciplines_by_semester.csv', index=False, encoding='utf-8-sig')
    df_schedule.to_csv(f'{output_dir}schedule.csv', index=False, encoding='utf-8-sig')

    if df_matrix is not None:
        df_matrix.to_csv(f'{output_dir}competence_matrix.csv', index=False, encoding='utf-8-sig')

    # Сохраняем аналитику, если она существует
    if 'load_by_sem' in locals():
        load_by_sem.to_csv(f'{output_dir}load_by_semester.csv', encoding='utf-8-sig')

    if 'load_by_course' in locals():
        load_by_course.to_csv(f'{output_dir}load_by_course.csv', encoding='utf-8-sig')

    if 'dept_stats' in locals() and len(dept_stats) > 0:
        dept_stats.to_csv(f'{output_dir}dept_stats.csv', encoding='utf-8-sig')

    if 'comp_stats' in locals() and len(comp_stats) > 0:
        comp_stats.to_csv(f'{output_dir}comp_stats.csv', encoding='utf-8-sig')

    print(f"Результаты сохранены в папку: {output_dir}")
    print("\nСохраненные файлы:")
    for f in os.listdir(output_dir):
        print(f"   - {f}")
else:
    print("Сохранение результатов отключено")

Сохранение результатов...
Результаты сохранены в папку: /content/drive/MyDrive/Curriculum_Parser_Results/

Сохраненные файлы:
   - disciplines.csv
   - disciplines_by_semester.csv
   - schedule.csv
   - competence_matrix.csv
   - load_by_semester.csv
   - load_by_course.csv
   - dept_stats.csv
   - comp_stats.csv
